# Finantal-LM — Pretraining (PyTorch from scratch, T4)

Self-contained: mounts Drive, clones the repo, installs deps, then trains. Checkpoints are written to Drive (`finantal_data/checkpoints/pretrain/`) and training **auto-resumes** from `latest.pt` if you reconnect.


In [ ]:
# Check the GPU (expect a Tesla T4)
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

In [ ]:
# Mount Google Drive (data + tokenizer + checkpoints live here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the code repo from GitHub
import os
REPO_URL = 'https://github.com/<YOUR_USERNAME>/finantal-lm.git'   # <-- EDIT
REPO_DIR = '/content/finantal-lm'
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull --ff-only
%cd $REPO_DIR

In [ ]:
# Install dependencies (torch already on Colab — not reinstalled)
!pip -q install -r requirements.txt

In [ ]:
# Point the code at the data on Drive. The default already matches this path,
# but we set it explicitly so it's obvious and overridable.
import os
os.environ['FINANTAL_DATA_ROOT'] = '/content/drive/MyDrive/finantal_data'
# Make the repo importable from notebook cells too
import sys; sys.path.insert(0, '/content/finantal-lm')
from config import paths as P
print(P.summary())

In [ ]:
# Verify the required assets exist on Drive before training
from config import paths as P
missing = P.verify_assets(require_tokenizer=True, require_pretrain=True, require_sft=True)
if missing:
    print('MISSING — upload these to Drive:')
    for m in missing: print('  -', m)
else:
    print('All assets present on Drive ✓')
    import json
    if os.path.exists(P.DATASET_STATS):
        print(json.dumps(json.load(open(P.DATASET_STATS)), ensure_ascii=False, indent=2))

## Train
Reads `config/train_config.json` → `pretrain`. Override any field with `--override key=value`. On CUDA OOM: lower `micro_batch_size`, raise `gradient_accumulation_steps` to keep the effective batch constant.


In [ ]:
!python -m training.pretrain --config config/train_config.json \
    --override micro_batch_size=8 gradient_accumulation_steps=16 max_steps=8000

### Safer low-memory setting


In [ ]:
# !python -m training.pretrain --override micro_batch_size=4 gradient_accumulation_steps=32 max_seq_len=1024

### Resume manually (auto-resume is on by default)


In [ ]:
# !python -m training.pretrain --override resume_from=/content/drive/MyDrive/finantal_data/checkpoints/pretrain/latest.pt

## Loss / perplexity curve


In [ ]:
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open('logs/pretrain_metrics.jsonl')]
plt.figure(figsize=(8,4)); plt.plot([r['step'] for r in rows], [r['loss'] for r in rows])
plt.xlabel('step'); plt.ylabel('loss'); plt.grid(True); plt.title('Pretrain loss'); plt.show()

## Generation sanity check


In [ ]:
!python -m training.sample --ckpt pretrain --prompt "التمويل هو" --max_new_tokens 80